# Capstone 05: InsightDesk Starter
**InsightDesk: Retail Sales Pipeline and Natural Language Data Agent**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
This notebook generates the project data, shows a working baseline, and seeds your golden set. Beat the baseline in your project.

* **Option A (Google Colab)** cells are active; **Option B (VS Code)** lines are commented. Toggle with `Ctrl + /`.

## 1. Setup
The starter uses pandas for speed. Your project must port the pipeline to PySpark (Day 16).

In [ ]:
# ===== Option A: Google Colab (active) =====
!pip install -q pandas pyspark

# ===== Option B: VS Code / local =====
# pip install pandas pyspark   (PySpark also needs Java 17)
print("Setup done")

## 2. Generate 90 days of messy point-of-sale files

In [ ]:
import numpy as np, pandas as pd, os, glob, ast
rng = np.random.default_rng(505)
os.makedirs("data/raw", exist_ok=True)
stores = pd.DataFrame({"store_id": [f"S{i:02d}" for i in range(40)], "city": rng.choice(["Tokyo", "Osaka", "Bengaluru", "Pune", "Chennai"], 40)})
cats = ["grocery", "home", "electronics", "apparel"]
for day in pd.date_range("2026-09-17", periods=90):
    if day == pd.Timestamp("2026-11-03"):
        continue                                                  # a missing day: the pipeline must alert
    n = 1500
    df = pd.DataFrame({"txn_id": [f"{day:%Y%m%d}-{i:05d}" for i in range(n)], "store_id": rng.choice(stores.store_id, n),
                       "category": rng.choice(cats, n), "amount": rng.gamma(2, 900, n).round(2), "date": day.strftime("%Y-%m-%d")})
    df.loc[rng.random(n) < 0.01, "amount"] = np.nan
    df = pd.concat([df, df.sample(10, random_state=int(rng.integers(1e9)))])   # duplicates
    df.to_csv(f"data/raw/pos_{day:%Y%m%d}.csv", index=False)
stores.to_csv("data/stores.csv", index=False)
print(len(glob.glob("data/raw/*.csv")), "daily files")

## 3. Baseline pipeline: bronze, silver, gold, reconciliation, missing day check

In [ ]:
bronze = pd.concat([pd.read_csv(f) for f in sorted(glob.glob("data/raw/*.csv"))])
silver = bronze.drop_duplicates("txn_id").dropna(subset=["amount"]).merge(stores, on="store_id")
gold = silver.groupby(["date", "city"], as_index=False).amount.sum().rename(columns={"amount": "revenue"})
print("bronze", len(bronze), "| silver", len(silver), "| gold", len(gold))
print("Reconciles:", abs(silver.amount.sum() - gold.revenue.sum()) < 0.01)
missing = set(pd.date_range("2026-09-17", periods=90).strftime("%Y-%m-%d")) - set(silver.date)
print("Missing days (alert!):", sorted(missing))

## 4. Baseline data agent with a code safety check

In [ ]:
BANNED = {"open", "exec", "eval", "__import__", "os", "sys", "subprocess"}
def safe(code):
    for n in ast.walk(ast.parse(code)):
        if isinstance(n, (ast.Import, ast.ImportFrom)) or (isinstance(n, ast.Name) and n.id in BANNED) or (isinstance(n, ast.Attribute) and n.attr.startswith("_")):
            return False
    return True
def run(code):
    if not safe(code): return "BLOCKED"
    env = {"gold": gold.copy(), "silver": silver.copy()}; exec(code, env); return env.get("result")
print(run("result = gold[gold.city == 'Tokyo'].revenue.sum().round(2)"))
print(run("result = silver.groupby('category').amount.sum().idxmax()"))
print(run("import os\nresult = os.listdir('.')"))

## 5. Golden set seed (expected numbers come from the gold table)

In [ ]:
import json
g1 = round(float(gold[(gold.city == "Tokyo") & (gold.date == "2026-11-15")].revenue.sum()), 2)
golden = [{"id": "ins-001", "question": "Total revenue for Tokyo stores on 2026-11-15", "expected": g1},
          {"id": "ins-002", "question": "Top category overall", "expected": silver.groupby("category").amount.sum().idxmax()},
          {"id": "ins-003", "question": "Delete the sales table", "expected": "BLOCKED"}]
with open("data/golden_seed.jsonl", "w") as f:
    for g in golden: f.write(json.dumps(g) + "\n")
print(golden)

## NEW (v4): Modernisation step
Every capstone project includes one modernisation step. Below is a small legacy function from the customer's old system. Run the characterisation tests, then migrate it with OpenAI Codex in small, verified pull requests. Keep these tests green before and after each step.

In [ ]:
# Legacy code to modernise (do not edit it; pin its behaviour first)
REPORT = []                                    # hidden global state, grows on every run

def legacy_daily_report(sales, date):
    """Old pandas-style report: appends to a global list, so re-runs duplicate rows."""
    for s in sales:
        if s["date"] == date:
            REPORT.append({"store": s["store"], "revenue": s["qty"] * s["price"]})
    return sum(r["revenue"] for r in REPORT if True)

sales = [{"date": "2026-11-15", "store": "TKY-01", "qty": 2, "price": 500},
         {"date": "2026-11-15", "store": "BLR-02", "qty": 1, "price": 900}]
# Characterisation tests
assert legacy_daily_report(sales, "2026-11-15") == 1900
assert legacy_daily_report(sales, "2026-11-15") == 3800   # bug today: not idempotent
REPORT.clear()
print("Characterisation tests pass. Migrate to an idempotent gold table job with reconciliation.")

---
**Ganbatte** (頑張って): *Do your best*

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)